<a href="https://colab.research.google.com/github/dataguirre/curso-ia-ciencia-de-datos/blob/main/sesion1/workshop1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fundamentos de IA para Ciencia de Datos

En este workshop, nos enfocaremos en profundizar en modelos black/white-box y modelos n-gramas.

## Actividad 1: Modelar una montaña rusa

### Objetivo

Escribir una función `modelar_montana_rusa(historial)` que reciba el historial de movimientos de una montaña rusa —una lista de strings con los valores `"sube"` o `"baja"`— y devuelva la probabilidad estimada de cada movimiento.

### Requisitos

1. La función recibe una lista de strings. Cada elemento es `"sube"` o `"baja"`.
2. La probabilidad de cada movimiento se estima con su **frecuencia relativa** en el historial:

   $$P(\text{movimiento}) = \frac{\text{veces que aparece}}{\text{total de movimientos}}$$

3. Devuelve un diccionario con la forma `{"sube": p1, "baja": p2}`, donde `p1 + p2 = 1`.
4. Si el historial está vacío, no hay información previa: se asume que ambos movimientos son igual de probables (0.5 y 0.5).
5. Si aparece un valor distinto de `"sube"` o `"baja"`, se lanza un error.

### Ejemplos

| Historial | Resultado esperado |
|---|---|
| `["sube", "sube", "baja", "baja"]` | `{"sube": 0.5, "baja": 0.5}` |
| `["sube", "sube", "sube", "baja"]` | `{"sube": 0.75, "baja": 0.25}` |
| `["baja"]` | `{"sube": 0.0, "baja": 1.0}` |
| `[]` | `{"sube": 0.5, "baja": 0.5}` |




In [ ]:
# TODO: completar función
def modelar_montana_rusa(historial: list) -> dict:
  return {"sube": 0, "baja": 0}

In [ ]:
# @title
# Celda de validación. No modificar
from math import isclose

def revisar(historial, esperado, nota=""):
    """Corre un caso y explica exactamente qué falló."""
    etiqueta = f"modelar_montana_rusa({historial!r})"
    if nota:
        etiqueta += f"   # {nota}"

    try:
        obtenido = modelar_montana_rusa(historial)
    except Exception as e:
        print(f"✗ {etiqueta}\n    lanzó {type(e).__name__}: {e}")
        return False

    fallas = []
    if not isinstance(obtenido, dict):
        fallas.append(f"debía devolver un dict, devolvió {type(obtenido).__name__}: {obtenido!r}")
    elif set(obtenido) != {"sube", "baja"}:
        fallas.append(f"las llaves debían ser {{'sube', 'baja'}}, fueron {set(obtenido)}")
    else:
        for movimiento in ("sube", "baja"):
            if not isclose(obtenido[movimiento], esperado[movimiento]):
                fallas.append(f"'{movimiento}': esperaba {esperado[movimiento]:.4g}, "
                              f"obtuvo {obtenido[movimiento]:.4g}")
        suma = obtenido["sube"] + obtenido["baja"]
        if not isclose(suma, 1.0):
            fallas.append(f"las probabilidades suman {suma:.4g} y deberían sumar 1")

    if fallas:
        print(f"✗ {etiqueta}")
        for falla in fallas:
            print(f"    → {falla}")
        return False

    print(f"✓ {etiqueta}\n    {obtenido}")
    return True


def revisar_error(historial, nota=""):
    """El historial es inválido: la función debe lanzar ValueError."""
    etiqueta = f"modelar_montana_rusa({historial!r})"
    if nota:
        etiqueta += f"   # {nota}"
    try:
        obtenido = modelar_montana_rusa(historial)
    except ValueError as e:
        print(f"✓ {etiqueta}\n    lanzó ValueError: {e}")
        return True
    except Exception as e:
        print(f"✗ {etiqueta}\n    → esperaba ValueError, lanzó {type(e).__name__}: {e}")
        return False
    print(f"✗ {etiqueta}\n    → esperaba ValueError, pero devolvió {obtenido}")
    return False


resultados = [
    revisar([], {"sube": 0.5, "baja": 0.5}, "sin historial: 50/50"),
    revisar(["sube"], {"sube": 1.0, "baja": 0.0}, "un solo dato"),
    revisar(["baja"], {"sube": 0.0, "baja": 1.0}, "un solo dato"),
    revisar(["sube", "baja"], {"sube": 0.5, "baja": 0.5}, "empate"),
    revisar(["sube", "sube", "sube", "baja"], {"sube": 0.75, "baja": 0.25}, "3 de 4 suben"),
    revisar(["baja", "baja", "baja", "sube"], {"sube": 0.25, "baja": 0.75}, "caso simétrico"),
    revisar(["sube", "baja", "baja"], {"sube": 1/3, "baja": 2/3}, "decimal periódico"),
    revisar(["sube"] * 7 + ["baja"] * 3, {"sube": 0.7, "baja": 0.3}, "historial largo"),
    revisar_error(["sube", "quieto"], "movimiento inválido"),
    revisar_error(["sube", "SUBE"], "sensible a mayúsculas"),
]

pasaron = sum(resultados)
print(f"\n{pasaron}/{len(resultados)} casos pasaron.")
if pasaron < len(resultados):
    print("Revisa los casos marcados con ✗.")

## Actividad 2: Modelo de lenguaje bigrama

### Objetivo

Construir en Google Colab un modelo **2-grama** que, dado un texto, prediga la siguiente palabra con su probabilidad, igual que las sugerencias del teclado del celular.

La idea del modelo es la misma de la Actividad 1 (contar y dividir), pero ahora el conteo está **condicionado al contexto**: en vez de preguntar "¿qué tan probable es 'sube'?", preguntamos "¿qué tan probable es 'cine' **dado que** la palabra anterior fue 'al'?".

$$P(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$

### El corpus

Usamos las oraciones en español de [Tatoeba](https://tatoeba.org): frases cortas y cotidianas, ideales para un modelo de siguiente palabra. El archivo `corpus_es.txt` tiene una oración por línea.

> Los datos provienen de Tatoeba y se distribuyen bajo licencia Creative Commons (CC-BY).

### Los cinco pasos

Son los mismos de la clase:

1. **Añadir marcadores** de inicio `<s>` y fin `</s>` a cada oración.
2. **Tokenizar**: partir la oración en palabras.
3. **Extraer n-gramas**: recorrer los tokens en parejas consecutivas.
4. **Contar** cuántas veces aparece cada pareja.
5. **Dividir** el conteo de la pareja entre el conteo del contexto.

### Qué debe entregar

Las cuatro funciones completas y las pruebas de la última celda en verde.


In [ ]:
# -- Cargar el corpus

# TODO: Cargue el corpus_es.txt a Google Colab y carguelo aquí

corpus = None # no cambie el nombre de la variable

In [ ]:
# -- Preparar texto

import re
from collections import defaultdict, Counter

PATRON = r"[a-záéíóúüñ]+"   # palabras en minúscula, sin puntuación ni números
INICIO, FIN = "<s>", "</s>"


def tokenizar(oracion):
    """
    Pasos 1 y 2: convierte una oración en una lista de tokens,
    en minúsculas, con los marcadores de inicio y fin.

    tokenizar("Quiero ir al cine")
        -> ["<s>", "quiero", "ir", "al", "cine", "</s>"]

    Pista: re.findall(PATRON, texto) devuelve la lista de palabras.
    """
    # TODO: pasar la oración a minúsculas y extraer las palabras con PATRON
    # TODO: devolver la lista con INICIO al principio y FIN al final
    pass

In [ ]:
# -- Entrenar el modelo

def entrenar_bigrama(corpus):
    """
    Pasos 3 y 4: cuenta cuántas veces cada palabra sigue a cada contexto.

    Devuelve un diccionario de contadores:
        bigramas["quiero"]  ->  Counter({"ir": 2, "comer": 1})

    Con el corpus ["quiero ir al cine", "quiero ir al parque", "quiero comer algo"]:
        bigramas["quiero"]["ir"] == 2
        bigramas["<s>"]["quiero"] == 3

    Pista: zip(tokens, tokens[1:]) recorre las parejas consecutivas.
    """
    bigramas = defaultdict(Counter)

    # TODO: para cada oración del corpus, tokenizarla
    # TODO: recorrer sus parejas (anterior, siguiente) y sumar 1 al contador

    return bigramas

In [ ]:
# -- Calcular probabilidades

def probabilidades(bigramas, contexto):
    """
    Paso 5: convierte los conteos de un contexto en probabilidades.

    probabilidades(bigramas, "quiero")  ->  {"ir": 0.67, "comer": 0.33}

    Reglas:
      - Las probabilidades deben sumar 1.
      - Si el contexto nunca se vio en el corpus, devuelve un diccionario vacío {}.
    """
    # TODO: obtener el Counter del contexto (ojo: puede no existir)
    # TODO: dividir cada conteo entre el total y devolver el diccionario
    pass

In [ ]:
# -- Predecir

def predecir(bigramas, texto, k=3):
    """
    Predice las k palabras más probables después de un texto.

    El contexto de un bigrama es SOLO la última palabra del texto:
        predecir(bigramas, "quiero ir al")  ->  usa el contexto "al"

    Devuelve una lista de tuplas (palabra, probabilidad) ordenada de mayor
    a menor probabilidad:
        [("cine", 0.5), ("parque", 0.5)]

    Casos especiales:
      - Si el texto está vacío, el contexto es INICIO ("<s>").
      - Si el contexto no se vio en el corpus, devuelve una lista vacía [].
    """
    # TODO: extraer las palabras del texto y tomar la última como contexto
    # TODO: pedir sus probabilidades y devolver las k mayores, ya ordenadas
    pass

In [ ]:
# @title
# Celda de validación. No modificar.
from math import isclose

CORPUS_PRUEBA = [
    "quiero ir al cine",
    "quiero ir al parque",
    "quiero comer algo",
]

fallas = []

def igual(a, b):
    """Compara resultados tolerando decimales (0.333... vs 1/3)."""
    if isinstance(a, float) or isinstance(b, float):
        return isinstance(a, (int, float)) and isinstance(b, (int, float)) and isclose(a, b)
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(igual(a[k], b[k]) for k in b)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(igual(x, y) for x, y in zip(a, b))
    return a == b


def revisar(descripcion, calcular, esperado):
    """calcular es una función sin argumentos, para poder atrapar los errores."""
    try:
        obtenido = calcular()
    except Exception as e:
        obtenido = f"{type(e).__name__}: {e}"

    if igual(obtenido, esperado):
        print(f"✓ {descripcion}")
    else:
        print(f"✗ {descripcion}\n    esperaba: {esperado}\n    obtuvo:   {obtenido}")
        fallas.append(descripcion)


b = entrenar_bigrama(CORPUS_PRUEBA)

revisar("tokenizar añade marcadores y pasa a minúsculas",
        lambda: tokenizar("Quiero ir al cine"),
        ["<s>", "quiero", "ir", "al", "cine", "</s>"])

revisar("'ir' sigue a 'quiero' dos veces", lambda: b["quiero"]["ir"], 2)
revisar("las 3 oraciones empiezan con 'quiero'", lambda: b["<s>"]["quiero"], 3)

revisar("probabilidades de 'quiero'",
        lambda: probabilidades(b, "quiero"),
        {"ir": 2/3, "comer": 1/3})

revisar("las probabilidades suman 1",
        lambda: isclose(sum(probabilidades(b, "al").values()), 1.0),
        True)

revisar("contexto nunca visto -> dict vacío",
        lambda: probabilidades(b, "jirafa"), {})

revisar("predecir después de 'quiero ir al'",
        lambda: predecir(b, "quiero ir al", k=2),
        [("cine", 0.5), ("parque", 0.5)])

revisar("texto vacío usa el contexto <s>",
        lambda: predecir(b, "", k=1),
        [("quiero", 1.0)])

revisar("contexto nunca visto -> lista vacía",
        lambda: predecir(b, "jirafa"), [])

print()
print("Todas las pruebas pasaron." if not fallas else f"Faltan {len(fallas)} pruebas: {fallas}")

In [ ]:
bigramas = entrenar_bigrama(corpus)
print(f"{len(bigramas):,} contextos distintos")

textos_de_prueba = [
    "quiero ir al cine",
    "quiero ir al parque",
    "quiero comer algo",
]

for texto in textos_de_prueba:
    print(f"\n{texto!r}")
    for palabra, p in predecir(bigramas, texto, k=5):
        print(f"   {palabra:<15} {p:.3f}")

### Preguntas para discutir

1. ¿Por qué hay contextos donde una palabra tiene probabilidad 1.00? ¿Confiarías en esa predicción?
2. ¿Qué pasa con `predecir` si escribes una palabra que no está en el corpus? ¿Cómo lo arreglaría un teclado real?
3. Si en vez de bigramas usaras trigramas, ¿las predicciones serían mejores? ¿Qué se pierde?

### Extensiones opcionales

- **Generar texto**: partir de `<s>` e ir escogiendo palabras con `random.choices()` usando las probabilidades, hasta llegar a `</s>`.
- **Suavizado de Laplace**: sumar 1 a todos los conteos para que ningún bigrama tenga probabilidad 0.
- **Trigramas**: cambiar el contexto de una palabra a una tupla de dos.